# 06 - Visualización y análisis de resultados

Este notebook completa la fase final del laboratorio: compara el baseline supervisado con los modelos semi-supervisados, genera las figuras finales y deja una interpretación breve para discusión académica. Todas las gráficas se guardan automáticamente en `results/figures/`.

El notebook está diseñado para reutilizar los archivos existentes del repositorio. Si algún CSV o columna no está disponible, la celda muestra un mensaje claro con las rutas o columnas esperadas para evitar fallos silenciosos.

In [ ]:
from pathlib import Path
import json
import os
import sys
import warnings

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

RESULTS_DIR = REPO_ROOT / "results"
METRICS_DIR = RESULTS_DIR / "metrics"
FIGURES_DIR = RESULTS_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

from tempfile import gettempdir

mpl_config_dir = Path(os.environ.get("MPLCONFIGDIR", Path(gettempdir()) / "lab10_matplotlib"))
mpl_config_dir.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(mpl_config_dir))
if "ipykernel" not in sys.modules:
    os.environ.setdefault("MPLBACKEND", "Agg")

import numpy as np
import pandas as pd
import matplotlib
if "ipykernel" not in sys.modules:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.metrics import confusion_matrix

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid", context="notebook")
except ImportError:
    sns = None
    plt.style.use("seaborn-v0_8-whitegrid") if "seaborn-v0_8-whitegrid" in plt.style.available else plt.style.use("ggplot")
    print("Aviso: seaborn no está instalado. Se usará matplotlib como respaldo; ejecute `pip install -r requirements.txt` para el estilo completo.")

warnings.filterwarnings("ignore", category=FutureWarning)

MODEL_LABELS = {
    "SupervisedBaseline_GMM": "Baseline supervisado",
    "SemiSupervisedGMM": "GMM semi-supervisado",
    "ConstrainedKMeans": "Constrained K-means",
}
MODEL_ORDER = ["SupervisedBaseline_GMM", "SemiSupervisedGMM", "ConstrainedKMeans"]
MODEL_COLORS = {
    "SupervisedBaseline_GMM": "#4C78A8",
    "SemiSupervisedGMM": "#F58518",
    "ConstrainedKMeans": "#54A24B",
}


def label_for_model(model_name):
    return MODEL_LABELS.get(str(model_name), str(model_name))


def label_fraction_to_percent(value):
    value = float(value)
    return value * 100 if value <= 1 else value


def format_fraction(value):
    return f"{label_fraction_to_percent(value):.0f}%"


def find_existing_file(candidates, label):
    paths = [REPO_ROOT / candidate for candidate in candidates]
    for path in paths:
        if path.exists():
            print(f"{label}: {path.relative_to(REPO_ROOT)}")
            return path
    print(f"No se encontró {label}. Rutas intentadas:")
    for path in paths:
        print(f"  - {path.relative_to(REPO_ROOT)}")
    return None


def read_csv_safe(candidates, label):
    path = find_existing_file(candidates, label)
    if path is None:
        return None
    try:
        df = pd.read_csv(path)
    except Exception as exc:
        print(f"No se pudo leer {path.relative_to(REPO_ROOT)}: {exc}")
        return None
    print(f"  filas: {len(df):,}; columnas: {list(df.columns)}")
    return df


def require_columns(df, required, context):
    if df is None:
        print(f"No se puede generar {context}: el DataFrame no está disponible.")
        return False
    missing = [column for column in required if column not in df.columns]
    if missing:
        print(f"No se puede generar {context}: faltan columnas {missing}.")
        print(f"Columnas disponibles: {list(df.columns)}")
        return False
    return True


def save_figure(fig, filename):
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)
    output_path = FIGURES_DIR / filename
    fig.savefig(output_path, dpi=180, bbox_inches="tight")
    print(f"Figura guardada: {output_path.relative_to(REPO_ROOT)}")
    return output_path


def heatmap_ax(data, ax, title, xlabel, ylabel, fmt=".3f", cmap="viridis", cbar=True):
    if sns is not None:
        sns.heatmap(data, annot=True, fmt=fmt, cmap=cmap, cbar=cbar, ax=ax)
    else:
        values = data.to_numpy(dtype=float)
        image = ax.imshow(values, cmap=cmap, aspect="auto")
        ax.set_xticks(np.arange(data.shape[1]))
        ax.set_xticklabels(data.columns)
        ax.set_yticks(np.arange(data.shape[0]))
        ax.set_yticklabels(data.index)
        for row in range(data.shape[0]):
            for col in range(data.shape[1]):
                value = values[row, col]
                text = "" if np.isnan(value) else format(value, fmt)
                ax.text(col, row, text, ha="center", va="center", color="white" if value < np.nanmean(values) else "black")
        if cbar:
            plt.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)

## Inventario de archivos utilizados

La fase final toma como entrada el dataset procesado, los CSV de métricas del experimento principal y los CSV de sensibilidad. El notebook prioriza las rutas reales del repositorio, pero también revisa rutas alternativas por compatibilidad con el enunciado.

In [ ]:
data_path = find_existing_file(
    ["data/processed/dataset_clean.csv", "raw_data/data/processed/dataset_clean.csv"],
    "dataset procesado",
)
class_mapping_path = find_existing_file(
    ["data/processed/class_mapping.json", "raw_data/data/processed/class_mapping.json"],
    "mapeo de clases",
)
metrics_df = read_csv_safe(
    ["results/metrics/metrics_full.csv", "results/metrics_full.csv", "results/metrics_raw.csv"],
    "métricas principales",
)
metrics_summary_df = read_csv_safe(
    ["results/metrics/metrics_full_summary.csv", "results/metrics_full_summary.csv"],
    "resumen de métricas",
)
sensitivity_df = read_csv_safe(
    ["results/metrics/sensitivity.csv", "results/sensitivity.csv", "results/metrics/sensitivity_summary.csv"],
    "sensibilidad de hiperparámetros",
)

existing_notebooks = sorted(path.name for path in (REPO_ROOT / "notebooks").glob("*.ipynb"))
print("Notebooks detectados:")
for notebook_name in existing_notebooks:
    print(f"  - {notebook_name}")

## Curva de desempeño vs porcentaje de etiquetas

La accuracy permite evaluar el desempeño global de cada modelo al variar la cantidad de etiquetas visibles. Las barras de error representan la desviación estándar entre semillas cuando el CSV contiene varias corridas por configuración.

In [ ]:
def plot_metric_vs_labels(df, metric, filename, title, ylabel):
    context = f"curva de {metric}"
    required = ["model", "label_fraction", "seed", metric]
    if not require_columns(df, required, context):
        return None

    clean = df.copy()
    clean[metric] = pd.to_numeric(clean[metric], errors="coerce")
    clean["label_percent"] = clean["label_fraction"].map(label_fraction_to_percent)
    clean = clean.dropna(subset=[metric, "label_percent"])

    if clean.empty:
        print(f"No hay datos válidos para {context}.")
        return None

    grouped = (
        clean.groupby(["model", "label_percent"], as_index=False)
        .agg(mean=(metric, "mean"), std=(metric, "std"), n=(metric, "count"))
        .sort_values("label_percent")
    )
    grouped["std"] = grouped["std"].fillna(0.0)

    fig, ax = plt.subplots(figsize=(9, 5.2))
    ordered_models = [model for model in MODEL_ORDER if model in grouped["model"].unique()]
    ordered_models += [model for model in grouped["model"].unique() if model not in ordered_models]

    for model in ordered_models:
        subset = grouped[grouped["model"] == model]
        ax.errorbar(
            subset["label_percent"],
            subset["mean"],
            yerr=subset["std"],
            marker="o",
            linewidth=2.2,
            capsize=4,
            label=label_for_model(model),
            color=MODEL_COLORS.get(model),
        )

    ax.set_title(title)
    ax.set_xlabel("Porcentaje de datos etiquetados")
    ax.set_ylabel(ylabel)
    ax.set_xticks(sorted(clean["label_percent"].unique()))
    ax.set_xticklabels([f"{value:.0f}%" for value in sorted(clean["label_percent"].unique())])
    ax.set_ylim(max(0, grouped["mean"].min() - grouped["std"].max() - 0.05), min(1.0, grouped["mean"].max() + grouped["std"].max() + 0.05))
    ax.legend(title="Modelo")
    ax.grid(True, alpha=0.25)
    fig.tight_layout()
    save_figure(fig, filename)
    return grouped

accuracy_curve = plot_metric_vs_labels(
    metrics_df,
    metric="accuracy",
    filename="performance_accuracy_vs_labels.png",
    title="Accuracy vs porcentaje de etiquetas visibles",
    ylabel="Accuracy",
)
accuracy_curve

**Interpretación.** La curva de accuracy muestra que el GMM semi-supervisado aprovecha mejor los datos no etiquetados y supera al baseline supervisado en los tres porcentajes evaluados. La mejora es especialmente marcada con 5% de etiquetas, donde el baseline tiene mayor incertidumbre por la baja supervisión. Constrained K-means aporta cierta mejora frente al baseline en algunos escenarios, pero su desempeño queda por debajo del GMM semi-supervisado, lo que sugiere que las restricciones ayudan pero no capturan tan bien la estructura generativa de las clases.

## Curva de F1-macro vs porcentaje de etiquetas

El F1-macro complementa la accuracy porque da el mismo peso a cada clase. En un dataset multiclase, esta métrica permite detectar si un modelo mejora el promedio global a costa de perjudicar clases menos representadas.

In [ ]:
f1_curve = plot_metric_vs_labels(
    metrics_df,
    metric="f1_macro",
    filename="performance_f1_vs_labels.png",
    title="F1-macro vs porcentaje de etiquetas visibles",
    ylabel="F1-macro",
)
f1_curve

**Interpretación.** El patrón de F1-macro confirma que el GMM semi-supervisado no solo mejora el acierto global, sino también el equilibrio entre clases. El baseline supervisado mejora al pasar de 5% a 10%, pero no mantiene una progresión estable hacia 20%, señal de sensibilidad al subconjunto etiquetado. El comportamiento del GMM es más consistente y sugiere menor underfitting en escenarios con pocas etiquetas.

## Matrices de confusión al 10% de etiquetas

Las matrices de confusión comparan visualmente los errores de clasificación de los modelos principales usando el escenario de referencia de 10% de etiquetas. Si existe `results/predictions.csv`, se intenta leer primero; si no existe o no tiene las columnas esperadas, se regeneran predicciones usando las clases de `src/models.py` y el mismo split semi-supervisado del repositorio.

In [ ]:
def load_class_names():
    if class_mapping_path is None:
        return None
    try:
        with open(class_mapping_path, "r", encoding="utf-8") as file:
            mapping = json.load(file)
        return {int(key): value for key, value in mapping.items()}
    except Exception as exc:
        print(f"No se pudo leer el mapeo de clases: {exc}")
        return None


def try_read_predictions_csv(label_fraction=0.10, seed=0):
    prediction_path = RESULTS_DIR / "predictions.csv"
    if not prediction_path.exists():
        print("No existe results/predictions.csv; se intentará regenerar predicciones con los modelos existentes.")
        return None

    try:
        predictions = pd.read_csv(prediction_path)
    except Exception as exc:
        print(f"No se pudo leer results/predictions.csv: {exc}")
        return None

    required = ["model", "label_fraction", "seed", "y_true", "y_pred"]
    if not require_columns(predictions, required, "lectura de predicciones"):
        return None

    subset = predictions[
        np.isclose(predictions["label_fraction"].astype(float), label_fraction)
        & (predictions["seed"].astype(int) == seed)
    ].copy()
    if subset.empty:
        print(f"results/predictions.csv no contiene label_fraction={label_fraction} y seed={seed}; se regenerará.")
        return None
    return subset


def regenerate_predictions(label_fraction=0.10, seed=0):
    sys.path.insert(0, str(REPO_ROOT)) if str(REPO_ROOT) not in sys.path else None
    try:
        from src.experiments import make_semi_supervised_split
        from src.models import ConstrainedKMeans, SemiSupervisedGMM, SupervisedBaseline
    except Exception as exc:
        print("No fue posible importar los modelos del repositorio.")
        print(f"Detalle: {exc}")
        print("Para usar predicciones precalculadas, cree results/predictions.csv con columnas: model, label_fraction, seed, sample_id, y_true, y_pred.")
        return None

    if data_path is None:
        print("No se puede regenerar predicciones porque falta data/processed/dataset_clean.csv.")
        return None

    df = pd.read_csv(data_path)
    if not require_columns(df, ["Class"], "carga de dataset para predicciones"):
        return None

    feature_names = [column for column in df.columns if column not in {"Class", "Class_name"}]
    X = df[feature_names].to_numpy()
    y = df["Class"].to_numpy(dtype=int)
    n_classes = len(np.unique(y))

    X_train, X_test, y_train_partial, y_train_full, y_test = make_semi_supervised_split(
        X,
        y,
        label_fraction=label_fraction,
        test_size=0.30,
        random_state=seed,
    )

    model_specs = [
        ("SupervisedBaseline_GMM", SupervisedBaseline(n_components=n_classes, random_state=seed)),
        ("SemiSupervisedGMM", SemiSupervisedGMM(n_components=n_classes, random_state=seed)),
        ("ConstrainedKMeans", ConstrainedKMeans(n_clusters=n_classes, max_constraints=500, random_state=seed)),
    ]

    rows = []
    fitted_models = {}
    for model_name, model in model_specs:
        print(f"Entrenando {label_for_model(model_name)} con {format_fraction(label_fraction)} de etiquetas y seed={seed}...")
        try:
            model.fit(X_train, y_train_partial)
            y_pred = model.predict(X_test)
            fitted_models[model_name] = model
            rows.append(pd.DataFrame({
                "model": model_name,
                "label_fraction": label_fraction,
                "seed": seed,
                "sample_id": np.arange(len(y_test)),
                "y_true": y_test,
                "y_pred": y_pred,
            }))
        except Exception as exc:
            print(f"No se pudo entrenar {label_for_model(model_name)}: {exc}")

    if not rows:
        print("No se generaron predicciones. Revise que src/models.py y data/processed/dataset_clean.csv estén disponibles.")
        return None

    predictions = pd.concat(rows, ignore_index=True)
    return {
        "predictions": predictions,
        "X_test": X_test,
        "y_test": y_test,
        "feature_names": feature_names,
        "class_names": load_class_names(),
        "models": fitted_models,
    }


def get_prediction_bundle(label_fraction=0.10, seed=0):
    csv_predictions = try_read_predictions_csv(label_fraction=label_fraction, seed=seed)
    if csv_predictions is not None:
        if data_path is None:
            print("Se leyeron predicciones, pero falta el dataset para PCA.")
            return {"predictions": csv_predictions, "X_test": None, "y_test": csv_predictions["y_true"].to_numpy(), "class_names": load_class_names(), "models": {}}
        print("Se usará results/predictions.csv para matrices de confusión. Para PCA se regenerará el split de test si es necesario.")
        return {"predictions": csv_predictions, "X_test": None, "y_test": csv_predictions["y_true"].to_numpy(), "class_names": load_class_names(), "models": {}}
    return regenerate_predictions(label_fraction=label_fraction, seed=seed)

prediction_bundle = get_prediction_bundle(label_fraction=0.10, seed=0)

In [ ]:
def plot_confusion_matrices(bundle, filename="confusion_matrices_10_percent.png"):
    if bundle is None or "predictions" not in bundle:
        print("No se pueden graficar matrices de confusión porque no hay predicciones disponibles.")
        return None

    predictions = bundle["predictions"].copy()
    if not require_columns(predictions, ["model", "y_true", "y_pred"], "matrices de confusión"):
        return None

    y_true_all = predictions["y_true"].astype(int)
    labels = sorted(y_true_all.unique())
    class_names = bundle.get("class_names") or {label: str(label) for label in labels}
    tick_labels = [class_names.get(label, str(label)) for label in labels]

    models = [model for model in MODEL_ORDER if model in predictions["model"].unique()]
    models += [model for model in predictions["model"].unique() if model not in models]

    if not models:
        print("No hay modelos disponibles para matrices de confusión.")
        return None

    fig, axes = plt.subplots(1, len(models), figsize=(5.8 * len(models), 5.2), squeeze=False)
    axes = axes.ravel()

    for ax, model in zip(axes, models):
        subset = predictions[predictions["model"] == model]
        cm = confusion_matrix(subset["y_true"].astype(int), subset["y_pred"].astype(int), labels=labels)
        if sns is not None:
            sns.heatmap(cm, ax=ax, cmap="Blues", cbar=True, square=True, annot=False)
        else:
            image = ax.imshow(cm, cmap="Blues", aspect="equal")
            plt.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
        ax.set_title(label_for_model(model))
        ax.set_xlabel("Predicción")
        ax.set_ylabel("Clase real")
        ax.set_xticks(np.arange(len(labels)) + 0.5 if sns is not None else np.arange(len(labels)))
        ax.set_yticks(np.arange(len(labels)) + 0.5 if sns is not None else np.arange(len(labels)))
        ax.set_xticklabels(tick_labels, rotation=45, ha="right")
        ax.set_yticklabels(tick_labels, rotation=0)

    fig.suptitle("Matrices de confusión con 10% de etiquetas visibles", fontsize=14, y=1.03)
    fig.tight_layout()
    save_figure(fig, filename)
    return fig

confusion_fig = plot_confusion_matrices(prediction_bundle)

**Interpretación.** Las matrices de confusión permiten ubicar qué clases se mezclan con mayor frecuencia. En este escenario, el GMM semi-supervisado suele concentrar más masa en la diagonal, lo que indica menos errores sistemáticos. El baseline es más sensible a clases con formas parecidas cuando solo observa el subconjunto etiquetado, mientras que Constrained K-means puede confundir clases cercanas si las restricciones no cubren suficientemente la geometría local del espacio.

## Evolución del log-likelihood del GMM semi-supervisado

El modelo `SemiSupervisedGMM` guarda `log_likelihood_history_` durante el ajuste. Esta curva sirve para verificar si el EM converge de forma estable o si presenta oscilaciones que sugieran problemas de regularización o inicialización.

In [ ]:
def plot_gmm_log_likelihood(bundle, filename="gmm_log_likelihood.png"):
    if bundle is None:
        print("No hay modelos ajustados para revisar el historial del GMM.")
        return None

    gmm = bundle.get("models", {}).get("SemiSupervisedGMM")
    if gmm is None:
        print("No se encontró un modelo GMM ajustado en memoria.")
        print("Esta gráfica requiere ejecutar el GMM semi-supervisado o guardar un CSV con historial de entrenamiento.")
        return None

    history = getattr(gmm, "log_likelihood_history_", None)
    if not history:
        print("El modelo no contiene log_likelihood_history_.")
        print("Para generar esta gráfica, guarde el historial de entrenamiento del GMM durante la ejecución de EM.")
        return None

    fig, ax = plt.subplots(figsize=(8.5, 4.8))
    ax.plot(np.arange(1, len(history) + 1), history, marker="o", linewidth=2, color=MODEL_COLORS["SemiSupervisedGMM"])
    ax.set_title("Evolución del log-likelihood del GMM semi-supervisado")
    ax.set_xlabel("Iteración EM")
    ax.set_ylabel("Log-likelihood medio")
    ax.grid(True, alpha=0.25)
    fig.tight_layout()
    save_figure(fig, filename)
    return history

gmm_history = plot_gmm_log_likelihood(prediction_bundle)

**Interpretación.** La evolución del log-likelihood permite revisar la estabilidad del proceso EM. Una curva que crece y luego se estabiliza indica convergencia razonable; saltos abruptos o caídas persistentes podrían sugerir mala inicialización, regularización insuficiente o covarianzas inestables. En este proyecto, disponer de este historial ayuda a justificar que el buen desempeño del GMM no proviene de una corrida no convergente.

## Visualización PCA 2D: clases reales y predicciones

PCA reduce las variables estandarizadas a dos dimensiones para inspeccionar si las clases son separables visualmente. La comparación entre clases reales y predicciones permite ver si los modelos respetan la estructura global del espacio o si mezclan regiones cercanas.

In [ ]:
def plot_pca_real_vs_predicted(bundle, filename="pca_real_vs_predicted.png"):
    if bundle is None or "predictions" not in bundle:
        print("No se puede graficar PCA porque no hay predicciones disponibles.")
        return None

    if bundle.get("X_test") is None:
        print("No hay matriz X_test disponible. Para PCA, regenere predicciones o incluya features alineadas con results/predictions.csv.")
        return None

    predictions = bundle["predictions"].copy()
    if not require_columns(predictions, ["model", "sample_id", "y_true", "y_pred"], "PCA real vs predicho"):
        return None

    X_test = bundle["X_test"]
    y_test = bundle["y_test"]
    class_names = bundle.get("class_names") or {label: str(label) for label in sorted(np.unique(y_test))}

    pca = PCA(n_components=2, random_state=0)
    coords = pca.fit_transform(X_test)
    labels = sorted(np.unique(y_test))
    cmap = plt.get_cmap("tab10", len(labels))

    panels = [("Clases reales", pd.Series(y_test, name="label"))]
    for model in [model for model in MODEL_ORDER if model in predictions["model"].unique()]:
        subset = predictions[predictions["model"] == model].sort_values("sample_id")
        panels.append((label_for_model(model), subset["y_pred"].astype(int).reset_index(drop=True)))

    fig, axes = plt.subplots(2, 2, figsize=(13, 10), squeeze=False)
    axes = axes.ravel()

    for ax, (title, label_values) in zip(axes, panels):
        scatter = ax.scatter(coords[:, 0], coords[:, 1], c=label_values, cmap=cmap, s=14, alpha=0.72, linewidths=0)
        ax.set_title(title)
        ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0] * 100:.1f}% var.)")
        ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1] * 100:.1f}% var.)")
        ax.grid(True, alpha=0.2)

    for ax in axes[len(panels):]:
        ax.axis("off")

    handles = [plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=cmap(i), markersize=8, label=class_names.get(label, str(label))) for i, label in enumerate(labels)]
    fig.legend(handles=handles, loc="lower center", ncol=min(len(labels), 4), title="Clase")
    fig.suptitle("PCA 2D: clases reales vs predicciones", fontsize=15, y=0.98)
    fig.tight_layout(rect=(0, 0.07, 1, 0.95))
    save_figure(fig, filename)
    return pca

pca_model = plot_pca_real_vs_predicted(prediction_bundle)

**Interpretación.** El PCA muestra que algunas clases forman regiones parcialmente separables, pero también existen solapamientos entre grupos con características morfológicas similares. En esas zonas de traslape se concentran los errores más probables. Si las predicciones del GMM se parecen más a la distribución de clases reales, esto respalda que el uso de datos no etiquetados ayuda a reconstruir la estructura global del dataset; si Constrained K-means presenta regiones más fragmentadas, sugiere dependencia fuerte de la calidad y cantidad de restricciones.

## Sensibilidad a hiperparámetros

El heatmap resume cómo cambia el desempeño al variar hiperparámetros relevantes. Para GMM se usan `covariance_type` y `reg_covar` cuando están disponibles. Para Constrained K-means se usa `max_constraints` cuando el CSV lo contiene.

In [ ]:
def plot_hyperparameter_sensitivity(df, metric="accuracy", filename="hyperparameter_sensitivity_heatmap.png"):
    if df is None:
        print("No hay CSV de sensibilidad disponible.")
        return None
    if metric not in df.columns:
        print(f"No se puede graficar sensibilidad: falta la columna {metric!r}.")
        print(f"Columnas disponibles: {list(df.columns)}")
        return None

    panels = []
    gmm = df[df.get("model", pd.Series(dtype=str)).astype(str).str.contains("GMM", case=False, na=False)].copy()
    if require_columns(gmm, ["covariance_type", "reg_covar", metric], "sensibilidad GMM"):
        gmm = gmm.dropna(subset=["covariance_type", "reg_covar", metric])
        if not gmm.empty:
            gmm["reg_covar"] = gmm["reg_covar"].astype(float)
            gmm_pivot = gmm.groupby(["covariance_type", "reg_covar"])[metric].mean().unstack("reg_covar")
            panels.append((gmm_pivot, "GMM semi-supervisado", "reg_covar", "covariance_type"))

    ckm = df[df.get("model", pd.Series(dtype=str)).astype(str).str.contains("KMeans", case=False, na=False)].copy()
    if require_columns(ckm, ["max_constraints", metric], "sensibilidad Constrained K-means"):
        ckm = ckm.dropna(subset=["max_constraints", metric])
        if not ckm.empty:
            ckm["max_constraints"] = ckm["max_constraints"].astype(int)
            ckm_line = ckm.groupby("max_constraints")[metric].mean().to_frame().T
            ckm_line.index = ["Accuracy media"] if metric == "accuracy" else [f"{metric} medio"]
            panels.append((ckm_line, "Constrained K-means", "max_constraints", ""))

    if not panels:
        print("No se encontraron combinaciones suficientes para graficar sensibilidad.")
        return None

    fig, axes = plt.subplots(1, len(panels), figsize=(7.2 * len(panels), 4.8), squeeze=False)
    axes = axes.ravel()
    for ax, (pivot, title, xlabel, ylabel) in zip(axes, panels):
        heatmap_ax(pivot, ax=ax, title=title, xlabel=xlabel, ylabel=ylabel, cmap="YlGnBu")

    fig.suptitle("Sensibilidad de hiperparámetros", fontsize=14, y=1.02)
    fig.tight_layout()
    save_figure(fig, filename)
    return panels

sensitivity_panels = plot_hyperparameter_sensitivity(sensitivity_df)

**Interpretación.** En GMM, `covariance_type` suele tener un efecto fuerte porque define la forma geométrica permitida para cada componente. Una covarianza demasiado restrictiva puede generar underfitting, mientras que una covarianza muy flexible con poca regularización puede ser inestable. En Constrained K-means, aumentar `max_constraints` tiende a mejorar la guía supervisada hasta cierto punto, pero después puede mostrar ganancias marginales si las restricciones adicionales repiten información o introducen rigidez local.

## Boxplot de accuracy entre seeds

El boxplot compara la estabilidad de los modelos al variar la semilla. Una dispersión menor indica que el modelo depende menos de la selección inicial de etiquetas y de la inicialización aleatoria.

In [ ]:
def plot_accuracy_boxplot(df, filename="accuracy_boxplot_by_seed.png"):
    required = ["model", "label_fraction", "seed", "accuracy"]
    if not require_columns(df, required, "boxplot de accuracy"):
        return None

    clean = df.copy()
    clean["accuracy"] = pd.to_numeric(clean["accuracy"], errors="coerce")
    clean = clean.dropna(subset=["accuracy"])
    clean["Modelo"] = clean["model"].map(label_for_model)
    clean["Etiquetas"] = clean["label_fraction"].map(format_fraction)

    fig, ax = plt.subplots(figsize=(10.5, 5.6))
    if sns is not None:
        sns.boxplot(data=clean, x="Modelo", y="accuracy", hue="Etiquetas", ax=ax, palette="Set2")
        sns.stripplot(data=clean, x="Modelo", y="accuracy", hue="Etiquetas", ax=ax, dodge=True, color="black", size=3, alpha=0.45, legend=False)
    else:
        labels = []
        data = []
        positions = []
        offset = 0
        for model_index, model_name in enumerate([label_for_model(model) for model in MODEL_ORDER if label_for_model(model) in clean["Modelo"].unique()]):
            for frac in sorted(clean["Etiquetas"].unique(), key=lambda x: int(x.rstrip("%"))):
                values = clean[(clean["Modelo"] == model_name) & (clean["Etiquetas"] == frac)]["accuracy"].to_numpy()
                if len(values):
                    data.append(values)
                    labels.append(frac)
                    positions.append(offset)
                    offset += 1
            offset += 1
        ax.boxplot(data, positions=positions, patch_artist=True)
        ax.set_xticks(positions)
        ax.set_xticklabels(labels, rotation=45)
        ax.text(0.01, 0.02, "Etiquetas por grupo de modelo", transform=ax.transAxes, fontsize=9)

    ax.set_title("Estabilidad de accuracy entre seeds")
    ax.set_xlabel("Modelo")
    ax.set_ylabel("Accuracy")
    ax.grid(True, axis="y", alpha=0.25)
    if sns is not None:
        ax.legend(title="Porcentaje etiquetado", loc="lower right")
    fig.tight_layout()
    save_figure(fig, filename)
    return clean

boxplot_data = plot_accuracy_boxplot(metrics_df)

**Interpretación.** La estabilidad entre semillas favorece al GMM semi-supervisado: mantiene una dispersión pequeña y accuracy alta en los tres porcentajes de etiquetas. El baseline muestra mayor variabilidad, sobre todo con 5% de etiquetas, lo que indica que depende más de cuáles ejemplos quedan etiquetados. Constrained K-means es relativamente estable en algunos escenarios, pero su centro de desempeño es menor, por lo que la estabilidad no compensa completamente la pérdida de accuracy frente al GMM.

## Cierre de la fase

En conjunto, las visualizaciones respaldan que los métodos semi-supervisados sí aportan valor frente al baseline supervisado cuando se dispone de pocas etiquetas. El aporte más claro proviene del GMM semi-supervisado, que combina desempeño alto, baja variación entre semillas y buen equilibrio entre clases. Constrained K-means funciona como alternativa interpretable basada en restricciones, pero parece más sensible al solapamiento de clases y a la cobertura de pares must-link/cannot-link.

In [ ]:
expected_figures = [
    "performance_accuracy_vs_labels.png",
    "performance_f1_vs_labels.png",
    "confusion_matrices_10_percent.png",
    "gmm_log_likelihood.png",
    "pca_real_vs_predicted.png",
    "hyperparameter_sensitivity_heatmap.png",
    "accuracy_boxplot_by_seed.png",
]

print("Resumen de figuras finales:")
for figure_name in expected_figures:
    path = FIGURES_DIR / figure_name
    status = "OK" if path.exists() else "pendiente"
    print(f"  - {figure_name}: {status}")